GDF-Net Teacher M4 (Asymmetric Focal-Tversky ConvFormer-S18) on ISIC-2018
Dataset: ISIC-2018 Task 1 (2,594 train / 100 validation / 1,000 test images at 384x384).
Protocol: Recall-oriented ConvFormer-S18 trained with Asymmetric Focal-Tversky loss (alpha=0.35, beta=0.65, gamma=4/3), used for weight-space model soup interpolation (0.70 Base + 0.30 Asym) and as Teacher M4 in online distillation (Section 3.3).


Asymmetric Focal-Tversky fine-tuning and weight-space interpolation evaluation.


In [1]:
"""
========================================================================================
Phase 27 (Proposal PA2): Single-Model 100% Pure-CNN Optimization via
  1. Warm-Started Fine-Tuning from SOTA ConvFormer-S18 (29.50M params)
  2. Asymmetric Focal-Tversky + Structure Composite Loss (alpha=0.30 FP, beta=0.70 FN, gamma=1.33)
     -> Penalizes False Negatives 2.33x harder than False Positives to cure under-segmentation
        on giant, low-contrast diffuse melanomas.
  3. 5-Group Layer-Wise Learning Rate Decay (LLRD) with Ultra-Gentle Low-Level Preservation
  4. Warm-Started ModelEMA (initialized directly at the SOTA theta_init weights)
  5. Weight-Space Ensembling (WiSE-FT: (1 - lambda*) * theta_init + lambda* * theta_asym)
     -> Produces a SINGLE 29.50M-param Pure-CNN model (1x inference cost, 1x checkpoint)
  6. Adaptive Lesion Thresholding (ALT) calibrated on 100 Official Validation Images
========================================================================================
"""

import os
import sys
import glob
import time
import copy
import json
import random
import zipfile
import shutil
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
import cv2

from scipy.ndimage import distance_transform_edt, binary_erosion

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import timm
import albumentations as A
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

BACKBONE_NAME = "convformer_s18.sail_in22k_ft_in1k_384"
MODEL_TAG = "ConvFormerS18_AsymTversky"
FINETUNE_EPOCHS = 20
IMG_SIZE = 384
BATCH_SIZE = 16


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)


# ====================================================================
# 1. Canonical GDF-Net Pure-CNN Architecture (29.50M Params)
# ====================================================================
class HierarchicalBackboneV4(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=False, in_chans=3, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            features_only=True,
            out_indices=(0, 1, 2, 3),
            drop_path_rate=drop_path_rate
        )
        self.channels = self.backbone.feature_info.channels()

    def get_channels(self):
        return tuple(self.channels)

    def forward(self, x):
        feats = self.backbone(x)
        return feats[0], feats[1], feats[2], feats[3]


class MultiScaleFrequencyFilter(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.in_channels = in_channels
        mid_channels = max(in_channels // reduction, 16)

        self.dw_conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw_conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw_conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)

        self.pw_low = nn.Sequential(
            nn.Conv2d(in_channels * 3, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.hf_gate = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        low1 = self.dw_conv1(x)
        low3 = self.dw_conv3(x)
        low5 = self.dw_conv5(x)
        x_low = self.pw_low(torch.cat([low1, low3, low5], dim=1))
        x_high = x - x_low
        gate = self.hf_gate(x_low)
        x_high_filtered = x_high * gate
        out = self.fusion(torch.cat([x_low, x_high_filtered], dim=1))
        return out + x


class MaskGuidedLGAG(nn.Module):
    def __init__(self, skip_channels, gating_channels, out_channels=None):
        super().__init__()
        out_channels = out_channels or skip_channels
        self.skip_channels = skip_channels
        self.gating_channels = gating_channels

        self.g_proj = nn.Sequential(
            nn.Conv2d(gating_channels, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels)
        )

        mid_ch = max(skip_channels // 2, 16)
        self.conv1x1 = nn.Conv2d(skip_channels + 1, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu = nn.ReLU(inplace=True)

        self.dw_dil1 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, dilation=1, groups=mid_ch, bias=False)
        self.dw_dil2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=2, dilation=2, groups=mid_ch, bias=False)
        self.bn_dw = nn.BatchNorm2d(mid_ch)

        self.attn_proj = nn.Sequential(
            nn.Conv2d(mid_ch, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels),
            nn.Sigmoid()
        )

        self.out_conv = nn.Sequential(
            nn.Conv2d(skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, skip, gating, mask_prob=None):
        B, C, H, W = skip.shape
        g_up = F.interpolate(gating, size=(H, W), mode='bilinear', align_corners=False)
        g_proj = self.g_proj(g_up)

        if mask_prob is not None:
            m_up = F.interpolate(mask_prob, size=(H, W), mode='bilinear', align_corners=False)
        else:
            m_up = torch.zeros((B, 1, H, W), device=skip.device, dtype=skip.dtype)

        feat = skip + g_proj
        feat_with_mask = torch.cat([feat, m_up], dim=1)

        x = self.relu(self.bn1(self.conv1x1(feat_with_mask)))
        x = self.relu(self.bn_dw(self.dw_dil1(x) + self.dw_dil2(x)))
        attn = self.attn_proj(x)

        gated_skip = skip * attn
        return self.out_conv(gated_skip)


class MultiScaleConvolutionalAttentionModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.dw1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)
        self.bn_dw = nn.BatchNorm2d(in_channels * 3)

        self.pw = nn.Sequential(
            nn.Conv2d(in_channels * 3, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        mid_ch = max(out_channels // 4, 16)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, mid_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv2d(out_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        d1 = self.dw1(x)
        d3 = self.dw3(x)
        d5 = self.dw5(x)
        dw_cat = F.relu(self.bn_dw(torch.cat([d1, d3, d5], dim=1)), inplace=True)
        feat = self.pw(dw_cat)
        feat = feat * self.ca(feat)
        feat = feat * self.sa(feat)
        return self.refine(feat)


class CascadedMSCAMDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.mscam = MultiScaleConvolutionalAttentionModule(in_channels + skip_channels, out_channels)
        self.head = nn.Conv2d(out_channels, 1, kernel_size=1)

    def forward(self, x, gated_skip):
        x_up = self.upsample(x)
        if x_up.shape[2:] != gated_skip.shape[2:]:
            x_up = F.interpolate(x_up, size=gated_skip.shape[2:], mode='bilinear', align_corners=False)
        fused = torch.cat([x_up, gated_skip], dim=1)
        out_feat = self.mscam(fused)
        pred_logits = self.head(out_feat)
        return out_feat, pred_logits


class CanonicalGDFNet(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=False, in_chans=3, decoder_channels=64, drop_path_rate=0.2):
        super().__init__()
        self.backbone_name = backbone_name
        self.decoder_channels = decoder_channels

        self.backbone = HierarchicalBackboneV4(
            backbone_name=backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            drop_path_rate=drop_path_rate
        )
        c1, c2, c3, c4 = self.backbone.get_channels()

        self.fdf1 = MultiScaleFrequencyFilter(c1)
        self.fdf2 = MultiScaleFrequencyFilter(c2)
        self.fdf3 = MultiScaleFrequencyFilter(c3)
        self.fdf4 = MultiScaleFrequencyFilter(c4)

        dec4_ch = decoder_channels * 4
        self.dec4_proj = nn.Sequential(
            nn.Conv2d(c4, dec4_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec4_ch),
            nn.ReLU(inplace=True)
        )
        self.head4 = nn.Conv2d(dec4_ch, 1, kernel_size=1)

        dec3_ch = decoder_channels * 3
        dec2_ch = decoder_channels * 2
        dec1_ch = decoder_channels

        self.gate3 = MaskGuidedLGAG(skip_channels=c3, gating_channels=dec4_ch, out_channels=c3)
        self.gate2 = MaskGuidedLGAG(skip_channels=c2, gating_channels=dec3_ch, out_channels=c2)
        self.gate1 = MaskGuidedLGAG(skip_channels=c1, gating_channels=dec2_ch, out_channels=c1)

        self.dec3 = CascadedMSCAMDecoderBlock(in_channels=dec4_ch, skip_channels=c3, out_channels=dec3_ch)
        self.dec2 = CascadedMSCAMDecoderBlock(in_channels=dec3_ch, skip_channels=c2, out_channels=dec2_ch)
        self.dec1 = CascadedMSCAMDecoderBlock(in_channels=dec2_ch, skip_channels=c1, out_channels=dec1_ch)

        self.final_refine = nn.Sequential(
            nn.Conv2d(dec1_ch, dec1_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec1_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(dec1_ch, 1, kernel_size=1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        f1, f2, f3, f4 = self.backbone(x)

        s1 = self.fdf1(f1)
        s2 = self.fdf2(f2)
        s3 = self.fdf3(f3)
        s4 = self.fdf4(f4)

        d4 = self.dec4_proj(s4)
        p4 = self.head4(d4)
        m4_prob = torch.sigmoid(p4)

        gated_s3 = self.gate3(s3, d4, m4_prob)
        d3, p3 = self.dec3(d4, gated_s3)
        m3_prob = torch.sigmoid(p3)

        gated_s2 = self.gate2(s2, d3, m3_prob)
        d2, p2 = self.dec2(d3, gated_s2)
        m2_prob = torch.sigmoid(p2)

        gated_s1 = self.gate1(s1, d2, m2_prob)
        d1, p1 = self.dec1(d2, gated_s1)

        p1_up = F.interpolate(p1, size=(H, W), mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=(H, W), mode='bilinear', align_corners=False)
        p3_up = F.interpolate(p3, size=(H, W), mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=(H, W), mode='bilinear', align_corners=False)

        d1_up = F.interpolate(d1, size=(H, W), mode='bilinear', align_corners=False)
        refined_logits = self.final_refine(d1_up)

        out_final = refined_logits + p1_up + p2_up + p3_up + p4_up

        if self.training:
            return {'logits': out_final, 'p1': p1_up, 'p2': p2_up, 'p3': p3_up, 'p4': p4_up}
        return out_final


class WarmStartedModelEMA:
    """EMA initialized directly at the pretrained SOTA weights so it never suffers from cold-start drift."""
    def __init__(self, model, decay=0.995):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, ema_v in self.ema.state_dict().items():
            model_v = msd[k].detach()
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
            else:
                ema_v.copy_(model_v)


# ====================================================================
# 2. Asymmetric Focal-Tversky + Structure Composite Loss (Proposal PA2)
# ====================================================================
class AsymmetricFocalTverskyStructureLoss(nn.Module):
    """
    Combines:
      1. Boundary-Weighted Structure Loss (wBCE + wIoU) to preserve razor-sharp boundaries.
      2. Asymmetric Focal-Tversky Loss (alpha=0.30 FP, beta=0.70 FN, gamma=1.33):
         Penalizes False Negatives 2.33x harder than False Positives and focuses gradients
         on hard under-segmented lesions via focal exponent gamma = 4/3.
    """
    def __init__(self, alpha=0.30, beta=0.70, gamma=1.3333, kernel_size=31, weight_factor=5.0, tversky_weight=1.2):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.weight_factor = weight_factor
        self.tversky_weight = tversky_weight
        self.avg_pool = nn.AvgPool2d(kernel_size=kernel_size, stride=1, padding=kernel_size // 2)

    def forward_single(self, pred_logits, targets):
        with torch.no_grad():
            local_mean = self.avg_pool(targets)
            boundary_diff = torch.abs(targets - local_mean)
            weight = 1.0 + self.weight_factor * boundary_diff

        # 1. Asymmetric Pos-Weighted BCE (pos_weight=1.6 boosts low-confidence lesion margins)
        pos_w = torch.tensor(1.6, device=pred_logits.device, dtype=pred_logits.dtype)
        bce = F.binary_cross_entropy_with_logits(pred_logits, targets, pos_weight=pos_w, reduction='none')
        wbce = (weight * bce).sum(dim=(2, 3)) / (weight.sum(dim=(2, 3)) + 1e-8)

        probs = torch.sigmoid(pred_logits)

        # 2. Boundary-Weighted IoU
        intersection = (weight * probs * targets).sum(dim=(2, 3))
        union = (weight * (probs + targets - probs * targets)).sum(dim=(2, 3))
        wiou = 1.0 - (intersection + 1.0) / (union + 1.0)

        # 3. Asymmetric Focal-Tversky Loss (alpha=0.30 FP, beta=0.70 FN)
        tp = (probs * targets).sum(dim=(2, 3))
        fp = (probs * (1.0 - targets)).sum(dim=(2, 3))
        fn = ((1.0 - probs) * targets).sum(dim=(2, 3))
        tversky_idx = (tp + 1.0) / (tp + self.alpha * fp + self.beta * fn + 1.0)
        focal_tversky = torch.pow(torch.clamp(1.0 - tversky_idx, min=1e-6), self.gamma)

        return (0.5 * (wbce + wiou) + self.tversky_weight * focal_tversky).mean()

    def forward(self, outputs, targets):
        if isinstance(outputs, dict):
            l_final = self.forward_single(outputs['logits'], targets)
            l_p1 = self.forward_single(outputs['p1'], targets)
            l_p2 = self.forward_single(outputs['p2'], targets)
            l_p3 = self.forward_single(outputs['p3'], targets)
            l_p4 = self.forward_single(outputs['p4'], targets)
            return l_final + 0.6 * l_p1 + 0.4 * l_p2 + 0.2 * l_p3 + 0.1 * l_p4
        return self.forward_single(outputs, targets)


def build_gentle_llrd_groups(model):
    g_stage0, g_stage1, g_stage2, g_stage3, g_decoder = [], [], [], [], []
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            if "stages_0" in name or "stages.0" in name or "stem" in name:
                g_stage0.append(param)
            elif "stages_1" in name or "stages.1" in name:
                g_stage1.append(param)
            elif "stages_2" in name or "stages.2" in name:
                g_stage2.append(param)
            else:
                g_stage3.append(param)
        else:
            g_decoder.append(param)

    return [
        {"params": g_stage0, "lr": 5.0e-6, "name": "bb_stage0"},
        {"params": g_stage1, "lr": 1.0e-5, "name": "bb_stage1"},
        {"params": g_stage2, "lr": 2.0e-5, "name": "bb_stage2"},
        {"params": g_stage3, "lr": 3.5e-5, "name": "bb_stage3"},
        {"params": g_decoder, "lr": 6.0e-5, "name": "decoder_heads"},
    ]


# ====================================================================
# 3. Dataset, Caching, Inference & Adaptive Lesion Thresholding (ALT)
# ====================================================================
class ISICFineTuneDataset(Dataset):
    def __init__(self, imgs, msks, is_train=False):
        self.imgs = imgs
        self.msks = msks
        self.is_train = is_train

        if is_train:
            self.transform = A.Compose([
                A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.65, 1.0), p=0.60),
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08), rotate=(-180, 180), shear=(-8, 8), p=0.50),
                A.OneOf([
                    A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.05, p=1.0),
                    A.RandomGamma(gamma_limit=(85, 115), p=1.0),
                    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
                ], p=0.45),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        else:
            self.transform = A.Compose([
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_COLOR)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.array(Image.open(self.imgs[idx]).convert('RGB'))

        msk = cv2.imread(self.msks[idx], cv2.IMREAD_GRAYSCALE)
        if msk is None:
            msk = np.array(Image.open(self.msks[idx]).convert('L'))
        msk = (msk > 127).astype(np.float32)

        augmented = self.transform(image=img, mask=msk)
        img = np.transpose(augmented['image'].astype(np.float32), (2, 0, 1))
        msk = np.expand_dims(augmented['mask'].astype(np.float32), axis=0)
        return torch.from_numpy(img).float(), torch.from_numpy(msk).float()


def prepare_fast_ssd_cache(imgs, msks, cache_dir, target_size=IMG_SIZE, desc="Dataset"):
    os.makedirs(cache_dir, exist_ok=True)
    cached_imgs, cached_msks = [], []
    need_build = False
    for im_p, mk_p in zip(imgs, msks):
        base_id = os.path.splitext(os.path.basename(im_p))[0]
        c_im = os.path.join(cache_dir, f"{base_id}.jpg")
        c_mk = os.path.join(cache_dir, f"{base_id}_mask.png")
        cached_imgs.append(c_im)
        cached_msks.append(c_mk)
        if not (os.path.exists(c_im) and os.path.exists(c_mk)):
            need_build = True

    if need_build:
        print(f"[*] Caching {desc} ({len(imgs)} pairs -> {target_size}x{target_size})...", flush=True)
        t_cache = time.time()
        for im_p, mk_p, c_im, c_mk in zip(imgs, msks, cached_imgs, cached_msks):
            if not (os.path.exists(c_im) and os.path.exists(c_mk)):
                img = cv2.imread(im_p, cv2.IMREAD_COLOR)
                if img is None:
                    img = cv2.cvtColor(np.array(Image.open(im_p).convert('RGB')), cv2.COLOR_RGB2BGR)
                msk = cv2.imread(mk_p, cv2.IMREAD_GRAYSCALE)
                if msk is None:
                    msk = np.array(Image.open(mk_p).convert('L'))
                img_r = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
                msk_r = cv2.resize(msk, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
                cv2.imwrite(c_im, img_r, [int(cv2.IMWRITE_JPEG_QUALITY), 96])
                cv2.imwrite(c_mk, msk_r)
        print(f"[+] Cache ready in {time.time() - t_cache:.1f}s!", flush=True)
    return cached_imgs, cached_msks


def compute_metrics_v4(p, t, compute_surface=True):
    p = p.astype(bool)
    t = t.astype(bool)
    tp = np.logical_and(p, t).sum()
    fp = np.logical_and(p, ~t).sum()
    fn = np.logical_and(~p, t).sum()
    tn = np.logical_and(~p, ~t).sum()

    dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)
    miou = tp / (tp + fp + fn + 1e-8)
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    prec = tp / (tp + fp + 1e-8)
    fpr  = fp / (fp + tn + 1e-8)

    hd95, asd, bf2 = 0.0, 0.0, 0.0
    if compute_surface and t.sum() > 0 and p.sum() > 0:
        bg = (t ^ binary_erosion(t)).astype(bool)
        bp = (p ^ binary_erosion(p)).astype(bool)
        if bg.sum() > 0 and bp.sum() > 0:
            dt_g = distance_transform_edt(~bg)
            dt_p = distance_transform_edt(~bp)
            dists = np.concatenate([dt_g[bp], dt_p[bg]])
            hd95 = float(np.percentile(dists, 95))
            asd  = float((dt_g[bp].mean() + dt_p[bg].mean()) / 2.0)
            prec_b = (dt_g[bp] <= 2).sum() / bp.sum()
            rec_b  = (dt_p[bg] <= 2).sum() / bg.sum()
            bf2 = float(2 * prec_b * rec_b / (prec_b + rec_b + 1e-8))

    return {
        'dice': float(dice), 'miou': float(miou), 'sensitivity': float(sens),
        'specificity': float(spec), 'precision': float(prec), 'fpr': float(fpr),
        'hd95': hd95, 'asd': asd, 'bf2': bf2,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn)
    }


def post_process_prediction(pred_bin):
    pred_u8 = (pred_bin > 0).astype(np.uint8)
    if pred_u8.sum() == 0:
        return pred_u8
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_u8, connectivity=8)
    if num_labels <= 2:
        cleaned = pred_u8
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        max_area = np.max(areas)
        cleaned = np.zeros_like(pred_u8)
        for label_idx, area in enumerate(areas, start=1):
            if area >= max_area * 0.08:
                cleaned[labels == label_idx] = 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)


@torch.no_grad()
def get_model_probs(model, imgs, mode="tta4"):
    B, C, H, W = imgs.shape
    if mode == "single":
        return torch.sigmoid(model(imgs))
    p1 = torch.sigmoid(model(imgs))
    p2 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[3]))), dims=[3])
    p3 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2]))), dims=[2])
    p4 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2, 3]))), dims=[2, 3])
    if mode == "tta4":
        return (p1 + p2 + p3 + p4) / 4.0
    elif mode == "mstta6":
        imgs_448 = F.interpolate(imgs, size=(448, 448), mode='bilinear', align_corners=False)
        p5 = F.interpolate(torch.sigmoid(model(imgs_448)), size=(H, W), mode='bilinear', align_corners=False)
        p6_448 = torch.flip(torch.sigmoid(model(torch.flip(imgs_448, dims=[3]))), dims=[3])
        p6 = F.interpolate(p6_448, size=(H, W), mode='bilinear', align_corners=False)
        return (p1 + p2 + p3 + p4 + p5 + p6) / 6.0
    else:
        raise ValueError(f"Unknown mode: {mode}")


@torch.no_grad()
def extract_model_probs(model, loader, device, mode="tta4"):
    model.eval()
    plist, mlist = [], []
    for imgs, msks in loader:
        imgs = imgs.to(device)
        with autocast('cuda'):
            p = get_model_probs(model, imgs, mode=mode)
        plist.append(p.cpu().numpy()[:, 0])
        mlist.append((msks > 0.5).numpy()[:, 0])
    return np.concatenate(plist, axis=0), np.concatenate(mlist, axis=0)


def calibrate_threshold(probs_np, msks_np):
    best_t, best_dice = 0.50, -1.0
    for t in np.linspace(0.35, 0.85, 51):
        preds = probs_np > t
        tp = np.logical_and(preds, msks_np).sum(axis=(1, 2))
        fp = np.logical_and(preds, ~msks_np).sum(axis=(1, 2))
        fn = np.logical_and(~preds, msks_np).sum(axis=(1, 2))
        d = ((2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)).mean()
        if d > best_dice:
            best_dice = float(d)
            best_t = float(t)
    return best_t, best_dice


def compute_alt_thresholds_batch(probs_np, t_base, dt_high=0.18, dt_mid=0.08):
    N, H, W = probs_np.shape
    total_px = float(H * W)
    t_per_image = np.full(N, t_base, dtype=np.float32)

    for i in range(N):
        pi = probs_np[i]
        area_50 = float((pi > 0.50).sum()) / total_px
        area_35 = float((pi > 0.35).sum())
        area_75 = float((pi > 0.75).sum())
        rho = area_35 / max(area_75, 1.0)

        if rho > 1.65 or area_50 > 0.45:
            t_per_image[i] = max(0.50, t_base - dt_high)
        elif rho > 1.32 or area_50 > 0.28:
            t_per_image[i] = max(0.55, t_base - dt_mid)
        else:
            t_per_image[i] = t_base

    return t_per_image


def calibrate_alt_on_val(probs_np, msks_np, t_base):
    best_dt_high, best_dt_mid = 0.0, 0.0
    _, base_dice = calibrate_threshold(probs_np, msks_np)
    best_dice = base_dice

    for dt_high in [0.00, 0.06, 0.10, 0.14, 0.18, 0.22]:
        for dt_mid in [0.00, 0.03, 0.06, 0.09, 0.12]:
            if dt_mid > dt_high:
                continue
            t_vec = compute_alt_thresholds_batch(probs_np, t_base, dt_high=dt_high, dt_mid=dt_mid)
            preds = probs_np > t_vec[:, None, None]
            tp = np.logical_and(preds, msks_np).sum(axis=(1, 2))
            fp = np.logical_and(preds, ~msks_np).sum(axis=(1, 2))
            fn = np.logical_and(~preds, msks_np).sum(axis=(1, 2))
            d = float(((2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)).mean())
            if d > best_dice:
                best_dice = d
                best_dt_high = dt_high
                best_dt_mid = dt_mid

    if best_dt_high == 0.0:
        t_vec = compute_alt_thresholds_batch(probs_np, t_base, dt_high=0.14, dt_mid=0.06)
        preds = probs_np > t_vec[:, None, None]
        tp = np.logical_and(preds, msks_np).sum(axis=(1, 2))
        fp = np.logical_and(preds, ~msks_np).sum(axis=(1, 2))
        fn = np.logical_and(~preds, msks_np).sum(axis=(1, 2))
        d_prior = float(((2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)).mean())
        if d_prior >= base_dice - 0.0005:
            best_dt_high, best_dt_mid, best_dice = 0.14, 0.06, d_prior

    return best_dt_high, best_dt_mid, best_dice


def evaluate_from_probs(probs_np, msks_np, thresh=0.50, apply_post_process=False):
    records = []
    tot_tp, tot_fp, tot_fn = 0, 0, 0
    N = len(probs_np)
    is_vector_thresh = isinstance(thresh, np.ndarray)

    for i in range(N):
        ti = float(thresh[i]) if is_vector_thresh else float(thresh)
        pred_bin = (probs_np[i] > ti).astype(np.uint8)
        if apply_post_process:
            pred_bin = post_process_prediction(pred_bin)

        m = compute_metrics_v4(pred_bin, msks_np[i], compute_surface=True)
        tot_tp += m['tp']
        tot_fp += m['fp']
        tot_fn += m['fn']
        records.append(m)

    df = pd.DataFrame(records)
    global_dice = float((2.0 * tot_tp) / (2.0 * tot_tp + tot_fp + tot_fn + 1e-8))
    global_iou  = float(tot_tp / (tot_tp + tot_fp + tot_fn + 1e-8))

    summary = []
    for col in ['dice', 'miou', 'sensitivity', 'specificity', 'precision', 'fpr', 'hd95', 'asd', 'bf2']:
        summary.append({
            'Metric': col,
            'Mean': float(df[col].mean()),
            'Std': float(df[col].std()),
            'Median': float(df[col].median()),
            'Min': float(df[col].min()),
            'Max': float(df[col].max())
        })
    j_vals = df['miou'].values
    j_thresh = float(np.where(j_vals >= 0.65, j_vals, 0.0).mean())
    failed = float((j_vals < 0.65).sum())
    summary.append({'Metric': 'j_thresh', 'Mean': j_thresh, 'Std': 0.0, 'Median': j_thresh, 'Min': 0.0, 'Max': 1.0})
    summary.append({'Metric': 'failed_cases_j_lt_0.65', 'Mean': failed, 'Std': 0.0, 'Median': failed, 'Min': 0.0, 'Max': float(len(df))})

    summary_df = pd.DataFrame(summary)
    return df, summary_df, global_dice, global_iou


def interpolate_state_dicts(sd_a, sd_b, lam):
    out = {}
    for k in sd_a.keys():
        va, vb = sd_a[k], sd_b[k]
        if va.dtype.is_floating_point:
            out[k] = (1.0 - lam) * va + lam * vb
        else:
            out[k] = vb if lam >= 0.5 else va
    return out


# ====================================================================
# 4. Main Training & Evaluation Pipeline
# ====================================================================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[+] Active Compute Device: {device}", flush=True)
    if torch.cuda.is_available():
        print(f"[+] GPU Hardware: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True

    OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results_asym_tversky"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_ROOT = "/kaggle/temp/isic-2018" if os.path.exists("/kaggle") else "./isic-2018"
    os.makedirs(DATA_ROOT, exist_ok=True)

    def find_dir_by_name(target_name, search_roots):
        for base in search_roots:
            if not os.path.exists(base):
                continue
            for root, dirs, files in os.walk(base):
                if os.path.basename(root).lower() == target_name.lower():
                    return root
        return None

    def _unzip_all(root):
        for z in glob.glob(os.path.join(root, "*.zip")):
            print(f"[*] Extracting: {os.path.basename(z)}...", flush=True)
            try:
                with zipfile.ZipFile(z) as f:
                    f.extractall(root)
                os.remove(z)
            except Exception as e:
                print(f"[!] Warning while unzipping {z}: {e}", flush=True)

    search_roots = ["/kaggle/input", DATA_ROOT]
    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth", search_roots)
    val_img_dir   = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir    = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    if not test_gt_dir:
        subprocess.run(["kaggle", "datasets", "download", "-d", "mfahimhossen/isic2018-test-groundtruth", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)
    if not (train_img_dir and test_img_dir):
        subprocess.run(["kaggle", "datasets", "download", "-d", "tschandl/isic2018-challenge-task1-data-segmentation", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)
    if not val_img_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_input.zip", "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Validation_Input.zip"], check=True)
        _unzip_all(DATA_ROOT)
    if not val_gt_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_gt.zip", "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Validation_GroundTruth.zip"], check=True)
        _unzip_all(DATA_ROOT)

    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth", search_roots)
    val_img_dir   = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir    = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    def isic_id(path):
        base = os.path.basename(path).replace("_segmentation", "")
        return os.path.splitext(base)[0]

    def pair_data(img_dir, gt_dir):
        imgs = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        msks = sorted(glob.glob(os.path.join(gt_dir, "*.png")))
        id_map = {isic_id(m): m for m in msks}
        paired_x, paired_y = [], []
        for im in imgs:
            imid = isic_id(im)
            if imid in id_map:
                paired_x.append(im)
                paired_y.append(id_map[imid])
        return paired_x, paired_y

    train_x_raw, train_y_raw = pair_data(train_img_dir, train_gt_dir)
    val_x_raw, val_y_raw     = pair_data(val_img_dir, val_gt_dir)
    test_x_raw, test_y_raw   = pair_data(test_img_dir, test_gt_dir)
    print(f"[+] Datasets -> Train: {len(train_x_raw)} | Official Val: {len(val_x_raw)} | Blind Test: {len(test_x_raw)}", flush=True)

    CACHE_ROOT = os.path.join(DATA_ROOT, "fast_ssd_cache")
    train_x, train_y = prepare_fast_ssd_cache(train_x_raw, train_y_raw, os.path.join(CACHE_ROOT, "train_384"), IMG_SIZE, desc="Train Cache")
    val_x, val_y     = prepare_fast_ssd_cache(val_x_raw, val_y_raw, os.path.join(CACHE_ROOT, "val_384"), IMG_SIZE, desc="Val Cache")
    test_x, test_y   = prepare_fast_ssd_cache(test_x_raw, test_y_raw, os.path.join(CACHE_ROOT, "test_384"), IMG_SIZE, desc="Test Cache")

    train_loader = DataLoader(ISICFineTuneDataset(train_x, train_y, is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(ISICFineTuneDataset(val_x, val_y, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(ISICFineTuneDataset(test_x, test_y, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Locate Warm-Start Checkpoint
    def find_file(fname, search_roots):
        for base in search_roots:
            if not os.path.exists(base):
                continue
            for root, dirs, files in os.walk(base):
                if fname in files:
                    return os.path.join(root, fname)
        return None

    ckpt_search = ["/kaggle/input", "./checkpoints", "./GDF-Net/results", "."]
    ckpt_init_path = find_file("GDFNet_ConvFormerS18_ISIC2018_best.pth", ckpt_search) or find_file("GDFNet_ConvFormerS18_300ep_best.pth", ckpt_search)
    assert ckpt_init_path, "Error: Pretrained ConvFormer-S18 checkpoint not found!"
    print(f"[+] Warm-Starting from Pure-CNN SOTA Checkpoint: {ckpt_init_path}", flush=True)

    model = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False, drop_path_rate=0.2).to(device)
    st_init = torch.load(ckpt_init_path, map_location=device, weights_only=False)
    if isinstance(st_init, dict) and 'model_state' in st_init:
        st_init = st_init['model_state']
    model.load_state_dict(st_init, strict=True)
    theta_init_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Evaluate Baseline Warm-Start Model on Val before Fine-Tuning
    val_init_mstta, val_msks = extract_model_probs(model, val_loader, device, mode="mstta6")
    t_init_val, d_init_val = calibrate_threshold(val_init_mstta, val_msks)
    print(f"[+] Baseline Warm-Start Model (Epoch 0) Val MS-TTA: T_opt = {t_init_val:.3f} | Val Dice = {d_init_val:.4f}", flush=True)

    # Extract Baseline Warm-Start Model on Blind Test for reference (Single-Model + ALT without retraining)
    test_init_mstta, test_msks = extract_model_probs(model, test_loader, device, mode="mstta6")

    # Initialize Warm-Started EMA, Optimizer, Scheduler, and Asymmetric Focal-Tversky Loss
    ema_helper = WarmStartedModelEMA(model, decay=0.995)
    criterion = AsymmetricFocalTverskyStructureLoss(alpha=0.30, beta=0.70, gamma=1.3333, tversky_weight=1.2)
    param_groups = build_gentle_llrd_groups(model)
    optimizer = optim.AdamW(param_groups, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6)
    scaler = GradScaler('cuda')

    best_val_cal_dice = -1.0
    best_asym_state_cpu = copy.deepcopy(theta_init_cpu)
    history = []

    print("\n=======================================================", flush=True)
    print(f"[*] STARTING {FINETUNE_EPOCHS}-EPOCH ASYMMETRIC FOCAL-TVERSKY FINE-TUNING (alpha=0.30, beta=0.70)", flush=True)
    print("=======================================================", flush=True)

    for epoch in range(1, FINETUNE_EPOCHS + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0

        for step, (imgs, msks) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)

            # Multi-scale batch on 25% of steps (448x448) to reinforce large-lesion receptive field
            if step % 4 == 0:
                imgs = F.interpolate(imgs, size=(448, 448), mode='bilinear', align_corners=False)
                msks = F.interpolate(msks, size=(448, 448), mode='nearest')

            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, msks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            ema_helper.update(model)
            running_loss += loss.item()

        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        # Evaluate both RAW and EMA on Official Val (4-View TTA + Calibrated Threshold)
        val_raw_probs, _ = extract_model_probs(model, val_loader, device, mode="tta4")
        t_raw, d_raw = calibrate_threshold(val_raw_probs, val_msks)

        val_ema_probs, _ = extract_model_probs(ema_helper.ema, val_loader, device, mode="tta4")
        t_ema, d_ema = calibrate_threshold(val_ema_probs, val_msks)

        ep_best = max(d_raw, d_ema)
        ep_src = "EMA" if d_ema >= d_raw else "RAW"
        dt = time.time() - t0
        cur_lr = optimizer.param_groups[-1]['lr']

        print(f"Epoch {epoch:02d}/{FINETUNE_EPOCHS:02d} [{dt:.1f}s] | Loss: {avg_loss:.4f} | DecLR: {cur_lr:.6f} | "
              f"Val Raw CalDice: {d_raw:.4f} (T={t_raw:.2f}) | Val EMA CalDice: {d_ema:.4f} (T={t_ema:.2f})", flush=True)

        if ep_best > best_val_cal_dice:
            best_val_cal_dice = ep_best
            chosen_m = ema_helper.ema if ep_src == "EMA" else model
            best_asym_state_cpu = {k: v.detach().cpu().clone() for k, v in chosen_m.state_dict().items()}
            torch.save(best_asym_state_cpu, os.path.join(OUTPUT_DIR, f"GDFNet_{MODEL_TAG}_best.pth"))
            print(f"    [*] New Best Asymmetric-Tversky Model ({ep_src}) Saved at Epoch {epoch} (Val CalDice: {best_val_cal_dice:.4f})", flush=True)

        history.append({
            'epoch': epoch, 'loss': avg_loss, 'lr': cur_lr,
            'val_raw_cal_dice': d_raw, 'val_ema_cal_dice': d_ema, 'best_val_cal_dice': best_val_cal_dice
        })

    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, f"training_history_{MODEL_TAG}.csv"), index=False)

    # -------------------------------------------------------------
    # Weight-Space Ensembling (WiSE-FT) Calibration on Official Val
    # -------------------------------------------------------------
    print("\n=======================================================", flush=True)
    print("[*] CALIBRATING WEIGHT-SPACE INTERPOLATION (WiSE-FT) ON 100 OFFICIAL VAL IMAGES", flush=True)
    print("=======================================================", flush=True)

    best_lam = 0.50
    best_wise_val_dice = -1.0
    wise_model = CanonicalGDFNet(backbone_name=BACKBONE_NAME, pretrained=False).to(device)

    for lam in [0.15, 0.25, 0.35, 0.50, 0.65, 0.80, 1.00]:
        sd_wise = interpolate_state_dicts(theta_init_cpu, best_asym_state_cpu, lam)
        wise_model.load_state_dict(sd_wise, strict=True)
        val_w_probs, _ = extract_model_probs(wise_model, val_loader, device, mode="mstta6")
        tw, dw = calibrate_threshold(val_w_probs, val_msks)
        print(f"    WiSE-FT lambda={lam:.2f} ((1-lam)*theta_init + lam*theta_asym): T_opt = {tw:.3f} | Val MS-TTA Dice = {dw:.4f}", flush=True)
        if dw > best_wise_val_dice:
            best_wise_val_dice = dw
            best_lam = lam

    # If tie or near-tie, prefer balanced interpolation lam in [0.35, 0.65]
    if best_lam == 1.00:
        sd_50 = interpolate_state_dicts(theta_init_cpu, best_asym_state_cpu, 0.50)
        wise_model.load_state_dict(sd_50, strict=True)
        val_50_probs, _ = extract_model_probs(wise_model, val_loader, device, mode="mstta6")
        _, d50 = calibrate_threshold(val_50_probs, val_msks)
        if d50 >= best_wise_val_dice - 0.0008:
            best_lam = 0.50

    print(f"[+] Selected Optimal WiSE-FT Single Model: lambda* = {best_lam:.2f} | Val Dice = {best_wise_val_dice:.4f}", flush=True)
    sd_wise_best = interpolate_state_dicts(theta_init_cpu, best_asym_state_cpu, best_lam)
    torch.save(sd_wise_best, os.path.join(OUTPUT_DIR, f"GDFNet_{MODEL_TAG}_WiSE_best.pth"))

    # -------------------------------------------------------------
    # Official 1,000 Blind Test Evaluation (Single Pure-CNN Model)
    # -------------------------------------------------------------
    print("\n=======================================================", flush=True)
    print("[*] EVALUATING SINGLE PURE-CNN MODELS ON 1,000 OFFICIAL BLIND TEST IMAGES", flush=True)
    print("=======================================================", flush=True)

    # Load Best Asymmetric Fine-Tuned Model
    model.load_state_dict(best_asym_state_cpu, strict=True)
    val_asym_sv, _     = extract_model_probs(model, val_loader, device, mode="single")
    val_asym_tta4, _   = extract_model_probs(model, val_loader, device, mode="tta4")
    val_asym_mstta6, _ = extract_model_probs(model, val_loader, device, mode="mstta6")

    test_asym_sv, _     = extract_model_probs(model, test_loader, device, mode="single")
    test_asym_tta4, _   = extract_model_probs(model, test_loader, device, mode="tta4")
    test_asym_mstta6, _ = extract_model_probs(model, test_loader, device, mode="mstta6")

    # Load WiSE-FT Single Model (0.50 theta_init + 0.50 theta_asym)
    sd_wise_50 = interpolate_state_dicts(theta_init_cpu, best_asym_state_cpu, 0.50)
    wise_model.load_state_dict(sd_wise_50, strict=True)
    val_wise_sv, _     = extract_model_probs(wise_model, val_loader, device, mode="single")
    val_wise_mstta6, _ = extract_model_probs(wise_model, val_loader, device, mode="mstta6")
    test_wise_sv, _     = extract_model_probs(wise_model, test_loader, device, mode="single")
    test_wise_mstta6, _ = extract_model_probs(wise_model, test_loader, device, mode="mstta6")

    configs = []

    # 1. Baseline Warm-Start Single Model (M1 300ep) + MS-TTA + ALT (0 retraining epochs)
    dth_init, dtm_init, _ = calibrate_alt_on_val(val_init_mstta, val_msks, t_init_val)
    t_alt_init = compute_alt_thresholds_batch(test_init_mstta, t_init_val, dt_high=dth_init, dt_mid=dtm_init)
    configs.append(("SingleModel_Baseline_S18_MSTTA_ALT", test_init_mstta, t_alt_init, True,
                    f"1. Baseline Single S18 (300ep) MS-TTA + ALT (T_base={t_init_val:.2f}, dT=-{dth_init:.2f}/-{dtm_init:.2f})"))

    # 2. Asymmetric Tversky Single Model — Single-View Calibrated (No TTA)
    t_asv, _ = calibrate_threshold(val_asym_sv, val_msks)
    configs.append(("SingleModel_AsymTversky_SV_Cal", test_asym_sv, t_asv, False,
                    f"2. Asymmetric-Tversky Single S18 Single-View (Calibrated T={t_asv:.2f}, No TTA)"))

    # 3. Asymmetric Tversky Single Model — Single-View + ALT (No TTA)
    dth_asv, dtm_asv, _ = calibrate_alt_on_val(val_asym_sv, val_msks, t_asv)
    t_alt_asv = compute_alt_thresholds_batch(test_asym_sv, t_asv, dt_high=dth_asv, dt_mid=dtm_asv)
    configs.append(("SingleModel_AsymTversky_SV_ALT", test_asym_sv, t_alt_asv, True,
                    f"3. Asymmetric-Tversky Single S18 Single-View + ALT (T_base={t_asv:.2f}, No TTA)"))

    # 4. Asymmetric Tversky Single Model — 4-View TTA Calibrated
    t_atta4, _ = calibrate_threshold(val_asym_tta4, val_msks)
    configs.append(("SingleModel_AsymTversky_TTA4_Cal", test_asym_tta4, t_atta4, False,
                    f"4. Asymmetric-Tversky Single S18 4-View TTA (Calibrated T={t_atta4:.2f})"))

    # 5. Asymmetric Tversky Single Model — 6-View MS-TTA Calibrated
    t_amstta, _ = calibrate_threshold(val_asym_mstta6, val_msks)
    configs.append(("SingleModel_AsymTversky_MSTTA6_Cal", test_asym_mstta6, t_amstta, False,
                    f"5. Asymmetric-Tversky Single S18 6-View MS-TTA (Calibrated T={t_amstta:.2f})"))

    # 6. Asymmetric Tversky Single Model — 6-View MS-TTA + ALT + LCC
    dth_ams, dtm_ams, _ = calibrate_alt_on_val(val_asym_mstta6, val_msks, t_amstta)
    t_alt_ams = compute_alt_thresholds_batch(test_asym_mstta6, t_amstta, dt_high=dth_ams, dt_mid=dtm_ams)
    configs.append(("SingleModel_AsymTversky_MSTTA6_ALT", test_asym_mstta6, t_alt_ams, True,
                    f"6. Asymmetric-Tversky Single S18 6-View MS-TTA + ALT (T_base={t_amstta:.2f}, dT=-{dth_ams:.2f}/-{dtm_ams:.2f})"))

    # 7. WiSE-FT Single Model (Weight-Space 0.5*theta_init + 0.5*theta_asym) — Single-View + ALT (No TTA)
    t_wsv, _ = calibrate_threshold(val_wise_sv, val_msks)
    dth_wsv, dtm_wsv, _ = calibrate_alt_on_val(val_wise_sv, val_msks, t_wsv)
    t_alt_wsv = compute_alt_thresholds_batch(test_wise_sv, t_wsv, dt_high=dth_wsv, dt_mid=dtm_wsv)
    configs.append(("SingleModel_WiSE_FT_SV_ALT", test_wise_sv, t_alt_wsv, True,
                    f"7. WiSE-FT Single S18 (Weight-Space 29.50M) Single-View + ALT (T_base={t_wsv:.2f}, No TTA)"))

    # 8. WiSE-FT Single Model (Weight-Space 0.5*theta_init + 0.5*theta_asym) — 6-View MS-TTA + ALT + LCC
    t_wms, _ = calibrate_threshold(val_wise_mstta6, val_msks)
    dth_wms, dtm_wms, _ = calibrate_alt_on_val(val_wise_mstta6, val_msks, t_wms)
    t_alt_wms = compute_alt_thresholds_batch(test_wise_mstta6, t_wms, dt_high=dth_wms, dt_mid=dtm_wms)
    configs.append(("SingleModel_WiSE_FT_MSTTA6_ALT", test_wise_mstta6, t_alt_wms, True,
                    f"8. WiSE-FT Single S18 (Weight-Space 29.50M) 6-View MS-TTA + ALT (T_base={t_wms:.2f}, dT=-{dth_wms:.2f}/-{dtm_wms:.2f})"))

    results_overview = []
    for tag, probs_matrix, thresh_val, apply_pp, desc in configs:
        print(f"\n==================== {desc} ====================", flush=True)
        raw_df, summary_df, glob_dice, glob_iou = evaluate_from_probs(
            probs_matrix, test_msks, thresh=thresh_val, apply_post_process=apply_pp
        )
        summary_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{tag}.csv"), index=False)
        raw_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{tag}.csv"), index=False)

        mean_dice = summary_df.loc[summary_df['Metric'] == 'dice', 'Mean'].values[0]
        med_dice  = summary_df.loc[summary_df['Metric'] == 'dice', 'Median'].values[0]
        miou      = summary_df.loc[summary_df['Metric'] == 'miou', 'Mean'].values[0]
        sens      = summary_df.loc[summary_df['Metric'] == 'sensitivity', 'Mean'].values[0]
        prec      = summary_df.loc[summary_df['Metric'] == 'precision', 'Mean'].values[0]
        spec      = summary_df.loc[summary_df['Metric'] == 'specificity', 'Mean'].values[0]
        hd95      = summary_df.loc[summary_df['Metric'] == 'hd95', 'Mean'].values[0]
        asd       = summary_df.loc[summary_df['Metric'] == 'asd', 'Mean'].values[0]
        failed    = int(summary_df.loc[summary_df['Metric'] == 'failed_cases_j_lt_0.65', 'Mean'].values[0])
        t_display = float(np.mean(thresh_val)) if isinstance(thresh_val, np.ndarray) else float(thresh_val)

        print(summary_df.to_string(index=False), flush=True)
        print(f"[+] Global Micro Dice: {glob_dice:.4f} | Global IoU: {glob_iou:.4f}", flush=True)

        results_overview.append({
            'Mode': tag,
            'Description': desc,
            'Mean Threshold': t_display,
            'Macro Dice': mean_dice,
            'Median Dice': med_dice,
            'Global Micro Dice': glob_dice,
            'mIoU': miou,
            'Sensitivity': sens,
            'Precision': prec,
            'Specificity': spec,
            'HD95': hd95,
            'ASD': asd,
            'Failed Cases (J<0.65)': failed
        })

    master_df = pd.DataFrame(results_overview)
    master_df.to_csv(os.path.join(OUTPUT_DIR, "master_single_pure_cnn_asym_tversky_overview.csv"), index=False)
    print("\n=======================================================", flush=True)
    print("[+] MASTER SINGLE-MODEL 100% PURE-CNN (ASYMMETRIC TVERSKY + WiSE-FT + ALT) RESULTS", flush=True)
    print("=======================================================", flush=True)
    print(master_df[['Mode', 'Macro Dice', 'Median Dice', 'Global Micro Dice', 'mIoU', 'Sensitivity', 'Precision', 'HD95', 'Failed Cases (J<0.65)']].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


[+] Active Compute Device: cuda
[+] GPU Hardware: Tesla T4 (VRAM: 15.64 GB)
[+] Datasets -> Train: 2594 | Official Val: 100 | Blind Test: 1000
[*] Caching Train Cache (2594 pairs -> 384x384)...
[+] Cache ready in 477.4s!
[*] Caching Val Cache (100 pairs -> 384x384)...
[+] Cache ready in 10.9s!
[*] Caching Test Cache (1000 pairs -> 384x384)...
[+] Cache ready in 188.0s!
[+] Warm-Starting from Pure-CNN SOTA Checkpoint: /kaggle/input/datasets/namnguynnnn/gdfnetv4-isic-pretrained/GDFNet_ConvFormerS18_ISIC2018_best.pth
[+] Baseline Warm-Start Model (Epoch 0) Val MS-TTA: T_opt = 0.740 | Val Dice = 0.9150

[*] STARTING 20-EPOCH ASYMMETRIC FOCAL-TVERSKY FINE-TUNING (alpha=0.30, beta=0.70)
Epoch 01/20 [236.8s] | Loss: 0.6657 | DecLR: 0.000060 | Val Raw CalDice: 0.9069 (T=0.85) | Val EMA CalDice: 0.9124 (T=0.85)
    [*] New Best Asymmetric-Tversky Model (EMA) Saved at Epoch 1 (Val CalDice: 0.9124)
Epoch 02/20 [147.2s] | Loss: 0.6564 | DecLR: 0.000059 | Val Raw CalDice: 0.9127 (T=0.85) | Val EMA 